In [4]:
import torch

# ============================================================
# COLUMN PARALLEL LINEAR LAYER SIMULATION
# We simulate 2 GPUs using 2 separate tensors
# ============================================================

# --- Setup ---
torch.manual_seed(42)

batch_size = 4
input_features = 512
output_features = 1024  # this is what gets split across GPUs

# Full weight matrix (in real life, this would never exist on one GPU)
# Shape: [input_features x output_features]
W_full = torch.randn(input_features, output_features)

# Full input (same input goes to BOTH GPUs in column parallel)
# Shape: [batch_size x input_features]
X = torch.randn(batch_size, input_features)

# ============================================================
# SPLIT: divide weight matrix column-wise across 2 GPUs
# GPU 1 gets left half of columns
# GPU 2 gets right half of columns
# ============================================================

half = output_features // 2  # = 512

W_gpu1 = W_full[:, :half]   # shape: [512 x 512]  → GPU 1's weight chunk
W_gpu2 = W_full[:, half:]   # shape: [512 x 512]  → GPU 2's weight chunk

print(f"Full weight shape:   {W_full.shape}")
print(f"GPU 1 weight shape:  {W_gpu1.shape}")
print(f"GPU 2 weight shape:  {W_gpu2.shape}")
print(f"Input shape:         {X.shape}")

# ============================================================
# COMPUTE: each GPU multiplies the FULL input by its weight chunk
# No communication needed here — fully independent work
# ============================================================

out_gpu1 = X @ W_gpu1   # shape: [batch x 512]
out_gpu2 = X @ W_gpu2   # shape: [batch x 512]

print(f"\nGPU 1 partial output shape: {out_gpu1.shape}")
print(f"GPU 2 partial output shape: {out_gpu2.shape}")

# ============================================================
# COMBINE: concatenate the two partial outputs side by side
# This is what happens after both GPUs finish their work
# ============================================================

output_parallel = torch.cat([out_gpu1, out_gpu2], dim=1)  # shape: [batch x 1024]

# ============================================================
# VERIFY: compare with full single-GPU computation
# Both should give identical results
# ============================================================

output_full = X @ W_full  # this is what a single GPU would compute

print(f"\nParallel output shape: {output_parallel.shape}")
print(f"Full output shape:     {output_full.shape}")

# Check if they match (they must match exactly)
match = torch.allclose(output_parallel, output_full, atol=1e-5)
print(f"\nDo they match? {match}")  # should print True
print(" Column parallel simulation correct!" if match else "❌ Something went wrong")

Full weight shape:   torch.Size([512, 1024])
GPU 1 weight shape:  torch.Size([512, 512])
GPU 2 weight shape:  torch.Size([512, 512])
Input shape:         torch.Size([4, 512])

GPU 1 partial output shape: torch.Size([4, 512])
GPU 2 partial output shape: torch.Size([4, 512])

Parallel output shape: torch.Size([4, 1024])
Full output shape:     torch.Size([4, 1024])

Do they match? True
 Column parallel simulation correct!


In [6]:
import torch

# ============================================================
# ROW PARALLEL LINEAR LAYER SIMULATION
# Here BOTH the weight AND the input are split
# At the end, partial results are ADDED (not concatenated)
# The addition across GPUs = all-reduce
# ============================================================

torch.manual_seed(42)

batch_size = 4
input_features = 512
output_features = 256

# Full weight matrix
# Shape: [input_features x output_features]
W_full = torch.randn(input_features, output_features)

# Full input
# Shape: [batch x input_features]
X = torch.randn(batch_size, input_features)

# ============================================================
# SPLIT: divide weight matrix ROW-wise across 2 GPUs
# AND split the input correspondingly
# GPU 1 handles the first half of input features
# GPU 2 handles the second half of input features
# ============================================================

half = input_features // 2  # = 256

# Weight split (row-wise)
W_gpu1 = W_full[:half, :]   # shape: [256 x 256]  → GPU 1
W_gpu2 = W_full[half:, :]   # shape: [256 x 256]  → GPU 2

# Input split (must match weight row split)
X_gpu1 = X[:, :half]        # shape: [batch x 256] → GPU 1
X_gpu2 = X[:, half:]        # shape: [batch x 256] → GPU 2

print(f"Full weight shape:    {W_full.shape}")
print(f"GPU 1 weight shape:   {W_gpu1.shape}")
print(f"GPU 2 weight shape:   {W_gpu2.shape}")
print(f"GPU 1 input shape:    {X_gpu1.shape}")
print(f"GPU 2 input shape:    {X_gpu2.shape}")

# ============================================================
# COMPUTE: each GPU multiplies its input chunk by its weight chunk
# This gives a PARTIAL SUM — not the full answer yet
# ============================================================

partial_gpu1 = X_gpu1 @ W_gpu1   # shape: [batch x 256]
partial_gpu2 = X_gpu2 @ W_gpu2   # shape: [batch x 256]

print(f"\nGPU 1 partial sum shape: {partial_gpu1.shape}")
print(f"GPU 2 partial sum shape: {partial_gpu2.shape}")

# ============================================================
# ALL-REDUCE: add the partial sums together
# In real multi-GPU systems, this is done via NCCL all-reduce
# Here we just add them manually to simulate it
# After all-reduce, BOTH GPUs would have this same final result
# ============================================================

output_allreduced = partial_gpu1 + partial_gpu2   # shape: [batch x 256]

# ============================================================
# VERIFY: compare with full single-GPU computation
# ============================================================

output_full = X @ W_full   # shape: [batch x 256]

match = torch.allclose(output_allreduced, output_full, atol=1e-5)
print(f"\nAll-reduced output shape: {output_allreduced.shape}")
print(f"Full output shape:        {output_full.shape}")
print(f"\nDo they match? {match}")
print("Row parallel + all-reduce simulation correct!" if match else "❌ Something went wrong")

Full weight shape:    torch.Size([512, 256])
GPU 1 weight shape:   torch.Size([256, 256])
GPU 2 weight shape:   torch.Size([256, 256])
GPU 1 input shape:    torch.Size([4, 256])
GPU 2 input shape:    torch.Size([4, 256])

GPU 1 partial sum shape: torch.Size([4, 256])
GPU 2 partial sum shape: torch.Size([4, 256])

All-reduced output shape: torch.Size([4, 256])
Full output shape:        torch.Size([4, 256])

Do they match? True
Row parallel + all-reduce simulation correct!


In [7]:
import torch
import torch.nn.functional as F
import math

# ============================================================
# ATTENTION HEAD PARTITIONING SIMULATION
# In a real 105B model there might be 96 heads
# We use 8 heads here for simplicity
# GPU 1 handles heads 0-3, GPU 2 handles heads 4-7
# ============================================================

torch.manual_seed(42)

batch_size = 2
seq_len = 16        # sequence length (number of tokens)
num_heads = 8       # total attention heads
head_dim = 64       # dimension per head
d_model = num_heads * head_dim   # = 512 total model dimension

# Full Q, K, V matrices for all heads
# Shape: [batch x seq_len x d_model]
Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

# ============================================================
# RESHAPE: split into individual heads
# Shape becomes: [batch x num_heads x seq_len x head_dim]
# ============================================================

def split_heads(x, num_heads, head_dim):
    # x shape: [batch x seq_len x d_model]
    batch, seq, _ = x.shape
    # reshape to [batch x seq x num_heads x head_dim]
    x = x.view(batch, seq, num_heads, head_dim)
    # transpose to [batch x num_heads x seq x head_dim]
    return x.transpose(1, 2)

Q_heads = split_heads(Q, num_heads, head_dim)   # [batch x 8 x 16 x 64]
K_heads = split_heads(K, num_heads, head_dim)
V_heads = split_heads(V, num_heads, head_dim)

print(f"Q with all heads shape: {Q_heads.shape}")

# ============================================================
# PARTITION: each GPU gets half the heads
# GPU 1: heads 0-3
# GPU 2: heads 4-7
# No communication needed — heads are independent of each other
# ============================================================

half_heads = num_heads // 2   # = 4

Q_gpu1 = Q_heads[:, :half_heads, :, :]   # [batch x 4 x 16 x 64]
K_gpu1 = K_heads[:, :half_heads, :, :]
V_gpu1 = V_heads[:, :half_heads, :, :]

Q_gpu2 = Q_heads[:, half_heads:, :, :]   # [batch x 4 x 16 x 64]
K_gpu2 = K_heads[:, half_heads:, :, :]
V_gpu2 = V_heads[:, half_heads:, :, :]

print(f"GPU 1 Q shape: {Q_gpu1.shape}  (handles heads 0-3)")
print(f"GPU 2 Q shape: {Q_gpu2.shape}  (handles heads 4-7)")

# ============================================================
# COMPUTE: each GPU runs full attention on its heads
# Completely independent — zero communication here
# ============================================================

def attention(Q, K, V):
    # Q,K,V shape: [batch x heads x seq x head_dim]
    scale = math.sqrt(Q.shape[-1])   # scale by sqrt(head_dim)

    # Step 1: compute QK^T scores
    # [batch x heads x seq x seq]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

    # Step 2: softmax over last dimension (over key positions)
    attn_weights = F.softmax(scores, dim=-1)

    # Step 3: weighted sum of values
    # [batch x heads x seq x head_dim]
    output = torch.matmul(attn_weights, V)
    return output

out_gpu1 = attention(Q_gpu1, K_gpu1, V_gpu1)   # [batch x 4 x 16 x 64]
out_gpu2 = attention(Q_gpu2, K_gpu2, V_gpu2)   # [batch x 4 x 16 x 64]

print(f"\nGPU 1 attention output shape: {out_gpu1.shape}")
print(f"GPU 2 attention output shape: {out_gpu2.shape}")

# ============================================================
# COMBINE: concatenate along the head dimension
# Then reshape back to [batch x seq x d_model]
# ============================================================

# Concatenate heads from both GPUs
out_combined = torch.cat([out_gpu1, out_gpu2], dim=1)   # [batch x 8 x 16 x 64]

# Reshape back to [batch x seq x d_model]
batch, heads, seq, hd = out_combined.shape
out_final = out_combined.transpose(1, 2).contiguous().view(batch, seq, heads * hd)

print(f"\nCombined output shape: {out_final.shape}")

# ============================================================
# VERIFY: compare against single GPU full attention
# ============================================================

out_single = attention(Q_heads, K_heads, V_heads)
out_single_flat = out_single.transpose(1, 2).contiguous().view(batch, seq, num_heads * head_dim)

match = torch.allclose(out_final, out_single_flat, atol=1e-5)
print(f"Do they match? {match}")
print(" Attention head partitioning simulation correct!" if match else "❌ Something went wrong")

Q with all heads shape: torch.Size([2, 8, 16, 64])
GPU 1 Q shape: torch.Size([2, 4, 16, 64])  (handles heads 0-3)
GPU 2 Q shape: torch.Size([2, 4, 16, 64])  (handles heads 4-7)

GPU 1 attention output shape: torch.Size([2, 4, 16, 64])
GPU 2 attention output shape: torch.Size([2, 4, 16, 64])

Combined output shape: torch.Size([2, 16, 512])
Do they match? True
 Attention head partitioning simulation correct!


In [5]:
import torch
import time

# ============================================================
# EXPERIMENT: measure simulated communication overhead
# We time how long all-reduce (the addition step) takes
# compared to the actual compute step
# This shows you WHY communication overhead matters
# ============================================================

torch.manual_seed(42)

# Simulate different sizes to see how overhead scales
sizes = [256, 512, 1024, 2048, 4096]

print(f"{'Output Size':<15} {'Compute Time (ms)':<22} {'AllReduce Time (ms)':<22} {'Ratio'}")
print("-" * 70)

for size in sizes:
    batch = 16
    input_dim = size
    output_dim = size

    W1 = torch.randn(input_dim, output_dim)
    W2 = torch.randn(input_dim, output_dim)
    X1 = torch.randn(batch, input_dim // 2)
    X2 = torch.randn(batch, input_dim // 2)

    # Time the compute (matrix multiply)
    runs = 100
    start = time.time()
    for _ in range(runs):
        p1 = X1 @ W1[:input_dim//2, :]
        p2 = X2 @ W2[input_dim//2:, :]
    compute_time = (time.time() - start) / runs * 1000  # ms

    # Time the all-reduce (addition of partial results)
    p1 = X1 @ W1[:input_dim//2, :]
    p2 = X2 @ W2[input_dim//2:, :]
    start = time.time()
    for _ in range(runs):
        result = p1 + p2   # this simulates all-reduce
    allreduce_time = (time.time() - start) / runs * 1000  # ms

    ratio = compute_time / allreduce_time if allreduce_time > 0 else float('inf')
    print(f"{size:<15} {compute_time:<22.4f} {allreduce_time:<22.4f} {ratio:.1f}x")

print("\nInterpretation:")
print("The ratio shows compute takes much longer than all-reduce")
print("This means tensor parallelism overhead is relatively small")
print("That's why it's efficient — communication cost is low vs compute")

Output Size     Compute Time (ms)      AllReduce Time (ms)    Ratio
----------------------------------------------------------------------
256             0.0381                 0.0050                 7.7x
512             0.1093                 0.0050                 22.1x
1024            0.4337                 0.0074                 58.7x
2048            2.0789                 0.0095                 218.6x
4096            17.5828                0.0204                 861.4x

Interpretation:
The ratio shows compute takes much longer than all-reduce
This means tensor parallelism overhead is relatively small
That's why it's efficient — communication cost is low vs compute
